# Notebook 10 — Virtual Screening Pipeline

**Project**: ML-Based QSAR Modeling for Anti-Leishmanial Sulfonamide Derivatives
**Input**: Trained models (RF, SVM, XGB, LGBM), scaler, selected features, training data
**Output**:
- `results/predictions/screening_all_predictions.csv` — All screening predictions
- `results/predictions/final_candidates_for_testing.csv` — Final diverse candidates
- `figures/nb10_virtual_screening.png` — Screening funnel summary

**Pipeline**:
1. Generate virtual sulfonamide library (combinatorial enumeration)
2. Compute descriptors (same pipeline as training)
3. Consensus prediction (>=3/4 models agree)
4. Applicability domain filter (kNN distance)
5. Lipinski + synthetic accessibility filters
6. Diversity selection (Tanimoto clustering)
7. Export final candidates

---


In [1]:
# ============================================================
# CELL 1: Imports and Configuration
# ============================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings, sys, joblib
from datetime import datetime
warnings.filterwarnings('ignore')

from rdkit import Chem
from rdkit.Chem import (
    Descriptors, rdMolDescriptors, AllChem, MACCSkeys,
    Draw, DataStructs, rdFingerprintGenerator
)
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.ML.Cluster import Butina
from rdkit.Chem import RDConfig
import os
sys.path.insert(0, os.path.join(RDConfig.RDContribDir, "SA_Score"))
import sascorer

from sklearn.neighbors import NearestNeighbors

import numpy as np
np.product = np.prod
from mordred import Calculator, descriptors as mordred_descriptors

SEED = 42
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
DATA = PROJECT_ROOT / 'data' / 'processed'
MODELS = PROJECT_ROOT / 'models'
FIGURES = PROJECT_ROOT / 'figures'
PRED_DIR = PROJECT_ROOT / 'results' / 'predictions'

for d in [FIGURES, PRED_DIR, PROJECT_ROOT / 'data' / 'external']:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M')}")


Project root: e:\PhD_Projecttttttttttttt\PhD_QSAR_Leishmania
Timestamp: 2026-05-23 00:27


In [2]:
# ============================================================
# CELL 2: Load Models, Scaler, Features, and Training Data
# ============================================================

rf   = joblib.load(MODELS / 'rf_classifier.joblib')
svm  = joblib.load(MODELS / 'svm_classifier.joblib')
xgb_model = joblib.load(MODELS / 'xgb_classifier.joblib')
lgbm_model = joblib.load(MODELS / 'lgbm_classifier.joblib')
scaler = joblib.load(MODELS / 'scaler.joblib')

models = {'RF': rf, 'SVM': svm, 'XGBoost': xgb_model, 'LightGBM': lgbm_model}

features = pd.read_csv(DATA / 'selected_features.csv')['feature'].tolist()
X_train_sel = pd.read_csv(DATA / 'X_train_selected.csv', index_col=0)

print(f"Models loaded: {list(models.keys())}")
print(f"Selected features: {len(features)}")
print(f"Training set size: {X_train_sel.shape[0]}")


Models loaded: ['RF', 'SVM', 'XGBoost', 'LightGBM']
Selected features: 1163
Training set size: 8627


## Step 1 — Generate Virtual Sulfonamide Library

We enumerate novel sulfonamide derivatives by combining core scaffolds with diverse R-groups
using RDKit reaction SMARTS. This creates a virtual library of compounds that share the
sulfonamide pharmacophore but explore diverse chemical space.


In [3]:
# ============================================================
# CELL 3: Generate Virtual Library from Training Data Scaffolds
# ============================================================

from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdmolops

# Load curated dataset to get active compound SMILES
curated = pd.read_csv(DATA / 'curated_dataset.csv')
train_set = pd.read_csv(DATA / 'train_set.csv')

# Get active training compounds
active_train = train_set[train_set['activity_class'] == 'Active']['std_smiles'].unique()
print(f'Active training compounds: {len(active_train)}')

# Strategy: Generate analogs by modifying active compounds
# 1. Take active compounds
# 2. Apply simple chemical modifications (halogen swap, methyl/methoxy addition)
# 3. This keeps compounds close to training chemical space

library = []
seen_smiles = set(active_train)  # Exclude known compounds

# Modification patterns: find-and-replace on SMILES
modifications = [
    ('F', 'Cl'),    # F -> Cl
    ('Cl', 'F'),    # Cl -> F
    ('Cl', 'Br'),   # Cl -> Br
    ('F', 'Br'),    # F -> Br
    ('OC', 'O'),    # remove methoxy -> hydroxyl
    ('(C)', '(F)'), # methyl -> fluoro
    ('(C)', '(Cl)'),# methyl -> chloro
    ('(F)', '(C)'), # fluoro -> methyl
    ('(O)', '(N)'), # hydroxyl -> amino
    ('(N)', '(O)'), # amino -> hydroxyl
]

# Apply modifications to top active compounds
n_analogs = 0
for smi in active_train[:500]:  # Use top 500 active compounds
    for old_frag, new_frag in modifications:
        if old_frag in smi:
            new_smi = smi.replace(old_frag, new_frag, 1)  # Replace first occurrence only
            mol = Chem.MolFromSmiles(new_smi)
            if mol is not None:
                canonical = Chem.MolToSmiles(mol)
                if canonical not in seen_smiles:
                    seen_smiles.add(canonical)
                    library.append({
                        'smiles': canonical,
                        'parent': smi,
                        'modification': f'{old_frag}->{new_frag}',
                    })
                    n_analogs += 1

# Also add Murcko scaffold derivatives
scaffold_counts = {}
for smi in active_train:
    mol = Chem.MolFromSmiles(smi)
    if mol:
        try:
            scaffold = MurckoScaffold.MakeScaffoldGeneric(
                MurckoScaffold.GetScaffoldForMol(mol)
            )
            scaffold_smi = Chem.MolToSmiles(scaffold)
            scaffold_counts[scaffold_smi] = scaffold_counts.get(scaffold_smi, 0) + 1
        except Exception:
            pass

print(f'Unique scaffolds in active compounds: {len(scaffold_counts)}')
print(f'Analogs generated: {n_analogs}')

screen_df = pd.DataFrame(library)
print(f'\nVirtual library: {len(screen_df)} novel compounds')
print(f'  (analogs of active training compounds via single-atom modifications)')


Active training compounds: 4853


[00:27:08] Explicit valence for atom # 4 O, 4, is greater than permitted
[00:27:08] Explicit valence for atom # 3 O, 4, is greater than permitted
[00:27:08] Explicit valence for atom # 2 O, 4, is greater than permitted
[00:27:08] Explicit valence for atom # 11 O, 4, is greater than permitted
[00:27:08] Explicit valence for atom # 8 O, 3, is greater than permitted
[00:27:08] Explicit valence for atom # 2 O, 3, is greater than permitted
[00:27:08] Explicit valence for atom # 8 O, 4, is greater than permitted
[00:27:08] Explicit valence for atom # 8 O, 3, is greater than permitted
[00:27:08] Explicit valence for atom # 8 O, 4, is greater than permitted
[00:27:08] Explicit valence for atom # 6 O, 4, is greater than permitted
[00:27:08] Explicit valence for atom # 1 O, 3, is greater than permitted
[00:27:08] Explicit valence for atom # 19 O, 4, is greater than permitted
[00:27:08] Explicit valence for atom # 1 O, 4, is greater than permitted
[00:27:08] Explicit valence for atom # 0 O, 3, is

[00:27:08] Explicit valence for atom # 1 O, 4, is greater than permitted
[00:27:08] Explicit valence for atom # 5 O, 4, is greater than permitted
[00:27:08] Explicit valence for atom # 8 O, 4, is greater than permitted
[00:27:08] Explicit valence for atom # 8 O, 4, is greater than permitted
[00:27:08] Explicit valence for atom # 0 O, 3, is greater than permitted
[00:27:08] Explicit valence for atom # 2 O, 4, is greater than permitted
[00:27:08] Explicit valence for atom # 4 O, 4, is greater than permitted
[00:27:08] Explicit valence for atom # 18 O, 4, is greater than permitted
[00:27:08] Explicit valence for atom # 11 O, 4, is greater than permitted
[00:27:08] Explicit valence for atom # 8 O, 3, is greater than permitted
[00:27:08] Explicit valence for atom # 3 O, 4, is greater than permitted
[00:27:08] Explicit valence for atom # 18 O, 4, is greater than permitted
[00:27:08] Explicit valence for atom # 17 O, 3, is greater than permitted
[00:27:08] Explicit valence for atom # 1 O, 4, 

[00:27:08] Explicit valence for atom # 1 O, 4, is greater than permitted


Unique scaffolds in active compounds: 1592
Analogs generated: 1028

Virtual library: 1028 novel compounds
  (analogs of active training compounds via single-atom modifications)


## Step 2 — Compute Descriptors (Same Pipeline as Training)

We must compute the exact same descriptors in the exact same order as notebook 03.


In [4]:
# ============================================================
# CELL 4: Compute Descriptors for Screening Library
# ============================================================

print("Computing descriptors for screening library...")
print("  Using the same pipeline as notebook 03 (Mordred + RDKit + ECFP6 + MACCS)\n")

# Parse molecules
mol_dict = {}
failed = []
for _, row in screen_df.iterrows():
    mol = Chem.MolFromSmiles(row['smiles'])
    if mol is not None:
        mol_dict[row['smiles']] = mol
    else:
        failed.append(row['smiles'])

print(f"Valid molecules: {len(mol_dict)} / {len(screen_df)}")
if failed:
    print(f"Failed to parse: {len(failed)}")

# --- Mordred 2D descriptors ---
print("\nComputing Mordred 2D descriptors...")
calc = Calculator(mordred_descriptors, ignore_3D=True)
mordred_results = {}
for smi, mol in mol_dict.items():
    result = calc(mol)
    mordred_results[smi] = [float(v) if not isinstance(v, (bool, type(None))) and hasattr(v, '__float__')
                            else (float(v) if isinstance(v, (int, float)) else np.nan)
                            for v in result]

mordred_cols = [f"mordred_{d.__class__.__name__}" if not hasattr(d, '__str__')
                else f"mordred_{str(d)}" for d in calc.descriptors]
mordred_df = pd.DataFrame.from_dict(mordred_results, orient='index', columns=mordred_cols)
print(f"  Mordred: {mordred_df.shape[1]} descriptors")

# --- RDKit 2D descriptors ---
print("Computing RDKit 2D descriptors...")
rdkit_desc_names = [d[0] for d in Descriptors.descList]
rdkit_calc = MoleculeDescriptors.MolecularDescriptorCalculator(rdkit_desc_names)

rdkit_results = {}
for smi, mol in mol_dict.items():
    vals = rdkit_calc.CalcDescriptors(mol)
    rdkit_results[smi] = list(vals)

rdkit_cols = [f"rdkit_{n}" for n in rdkit_desc_names]
rdkit_df = pd.DataFrame.from_dict(rdkit_results, orient='index', columns=rdkit_cols)
print(f"  RDKit: {rdkit_df.shape[1]} descriptors")

# --- ECFP6 fingerprints ---
print("Computing ECFP6 fingerprints (radius=3, 4096 bits)...")
ecfp6_results = {}
for smi, mol in mol_dict.items():
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=3, nBits=4096)
    ecfp6_results[smi] = list(fp)

ecfp6_cols = [f"ECFP6_{i}" for i in range(4096)]
ecfp6_df = pd.DataFrame.from_dict(ecfp6_results, orient='index', columns=ecfp6_cols)
print(f"  ECFP6: {ecfp6_df.shape[1]} bits")

# --- MACCS keys ---
print("Computing MACCS keys...")
maccs_results = {}
for smi, mol in mol_dict.items():
    fp = MACCSkeys.GenMACCSKeys(mol)
    maccs_results[smi] = list(fp)

maccs_cols = [f"MACCS_{i}" for i in range(167)]
maccs_df = pd.DataFrame.from_dict(maccs_results, orient='index', columns=maccs_cols)
print(f"  MACCS: {maccs_df.shape[1]} keys")

# --- Combine ---
all_desc = pd.concat([mordred_df, rdkit_df, ecfp6_df, maccs_df], axis=1)
all_desc = all_desc.replace([np.inf, -np.inf], np.nan)
all_desc = all_desc.fillna(0)
print(f"\nCombined descriptors: {all_desc.shape}")


Computing descriptors for screening library...
  Using the same pipeline as notebook 03 (Mordred + RDKit + ECFP6 + MACCS)



Valid molecules: 1028 / 1028

Computing Mordred 2D descriptors...


  Mordred: 1613 descriptors
Computing RDKit 2D descriptors...


  RDKit: 217 descriptors
Computing ECFP6 fingerprints (radius=3, 4096 bits)...


[00:31:22] DEPRECATION WARNING: please use MorganGenerator
[00:31:22] DEPRECATION WARNING: please use MorganGenerator
[00:31:22] DEPRECATION WARNING: please use MorganGenerator
[00:31:22] DEPRECATION WARNING: please use MorganGenerator
[00:31:22] DEPRECATION WARNING: please use MorganGenerator
[00:31:22] DEPRECATION WARNING: please use MorganGenerator
[00:31:22] DEPRECATION WARNING: please use MorganGenerator
[00:31:22] DEPRECATION WARNING: please use MorganGenerator
[00:31:22] DEPRECATION WARNING: please use MorganGenerator
[00:31:22] DEPRECATION WARNING: please use MorganGenerator
[00:31:22] DEPRECATION WARNING: please use MorganGenerator
[00:31:22] DEPRECATION WARNING: please use MorganGenerator
[00:31:22] DEPRECATION WARNING: please use MorganGenerator
[00:31:22] DEPRECATION WARNING: please use MorganGenerator
[00:31:22] DEPRECATION WARNING: please use MorganGenerator
[00:31:22] DEPRECATION WARNING: please use MorganGenerator
[00:31:22] DEPRECATION WARNING: please use MorganGenerat

...

[00:31:26] DEPRECATION WARNING: please use MorganGenerator
[00:31:26] DEPRECATION WARNING: please use MorganGenerator
[00:31:26] DEPRECATION WARNING: please use MorganGenerator
[00:31:26] DEPRECATION WARNING: please use MorganGenerator
[00:31:26] DEPRECATION WARNING: please use MorganGenerator
[00:31:26] DEPRECATION WARNING: please use MorganGenerator
[00:31:26] DEPRECATION WARNING: please use MorganGenerator
[00:31:26] DEPRECATION WARNING: please use MorganGenerator
[00:31:26] DEPRECATION WARNING: please use MorganGenerator
[00:31:26] DEPRECATION WARNING: please use MorganGenerator
[00:31:26] DEPRECATION WARNING: please use MorganGenerator
[00:31:26] DEPRECATION WARNING: please use MorganGenerator
[00:31:26] DEPRECATION WARNING: please use MorganGenerator
[00:31:26] DEPRECATION WARNING: please use MorganGenerator
[00:31:26] DEPRECATION WARNING: please use MorganGenerator
[00:31:26] DEPRECATION WARNING: please use MorganGenerator
[00:31:26] DEPRECATION WARNING: please use MorganGenerat

[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerat

[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerator
[00:31:27] DEPRECATION WARNING: please use MorganGenerat

  ECFP6: 4096 bits
Computing MACCS keys...


  MACCS: 167 keys

Combined descriptors: (1028, 6093)


In [5]:
# ============================================================
# CELL 5: Scale and Select Features (Match Training Pipeline)
# ============================================================

# The scaler was fitted on ALL descriptors (2111 features) in nb04.
# We must scale using ALL features first, THEN select the 1163.

# Get the full feature list the scaler was fitted on
scaler_features = scaler.feature_names_in_ if hasattr(scaler, 'feature_names_in_') else None

if scaler_features is not None:
    print(f'Scaler was fitted on {len(scaler_features)} features')
    # Create full descriptor matrix matching scaler features
    X_full = pd.DataFrame(0.0, index=all_desc.index, columns=scaler_features)
    common_cols = [c for c in scaler_features if c in all_desc.columns]
    X_full[common_cols] = all_desc[common_cols].values
    print(f'  Matched {len(common_cols)} / {len(scaler_features)} features from screening descriptors')
    
    # Scale using the full scaler
    X_full_scaled = pd.DataFrame(
        scaler.transform(X_full), columns=scaler_features, index=X_full.index
    )
    
    # Now select only the 1163 features used in modeling
    available_selected = [f for f in features if f in X_full_scaled.columns]
    X_screen_scaled = X_full_scaled[available_selected].copy()
else:
    # Fallback: scaler has no feature names (numpy-fitted)
    print('Scaler has no feature names, using numpy arrays')
    descriptor_names = pd.read_csv(DATA / 'descriptor_names.csv')['feature'].tolist()
    X_full = pd.DataFrame(0.0, index=all_desc.index, columns=descriptor_names)
    common_cols = [c for c in descriptor_names if c in all_desc.columns]
    X_full[common_cols] = all_desc[common_cols].values
    print(f'  Matched {len(common_cols)} / {len(descriptor_names)} features')
    
    X_full_scaled_np = scaler.transform(X_full.values)
    X_full_scaled = pd.DataFrame(X_full_scaled_np, columns=descriptor_names, index=X_full.index)
    X_screen_scaled = X_full_scaled[features].copy()

print(f'\nX_screen_scaled shape: {X_screen_scaled.shape}')
print(f'NaN check: {X_screen_scaled.isna().sum().sum()} NaNs')


Scaler was fitted on 2111 features


  Matched 2111 / 2111 features from screening descriptors

X_screen_scaled shape: (1028, 1163)
NaN check: 0 NaNs


## Step 3 — Consensus Prediction

A compound is predicted Active only if >=3 out of 4 models agree.


In [6]:
# ============================================================
# CELL 6: Consensus Prediction
# ============================================================

print("Running consensus prediction (4 models)...\n")

pred_df = screen_df[screen_df['smiles'].isin(X_screen_scaled.index)].copy()
pred_df = pred_df.set_index('smiles')

# Individual model predictions
for name, model in models.items():
    X_input = X_screen_scaled.loc[pred_df.index]
    pred_df[f'{name}_pred'] = model.predict(X_input)
    pred_df[f'{name}_prob'] = model.predict_proba(X_input)[:, 1]
    n_active = pred_df[f'{name}_pred'].sum()
    print(f"  {name}: {n_active} predicted Active ({n_active/len(pred_df)*100:.1f}%)")

# Consensus (>=3/4 agree)
pred_cols = [f'{name}_pred' for name in models.keys()]
pred_df['votes'] = pred_df[pred_cols].sum(axis=1)
pred_df['consensus_pred'] = (pred_df['votes'] >= 3).astype(int)

prob_cols = [f'{name}_prob' for name in models.keys()]
pred_df['avg_probability'] = pred_df[prob_cols].mean(axis=1)
pred_df['confidence'] = pred_df['votes'] / len(models)

n_consensus = pred_df['consensus_pred'].sum()
print(f"\nConsensus hits (>=3/4): {n_consensus} / {len(pred_df)} ({n_consensus/len(pred_df)*100:.1f}%)")

# Save all predictions
pred_df.to_csv(PRED_DIR / 'screening_all_predictions.csv')
print(f"Saved: results/predictions/screening_all_predictions.csv")


Running consensus prediction (4 models)...



  RF: 875 predicted Active (85.1%)


  SVM: 987 predicted Active (96.0%)


  XGBoost: 930 predicted Active (90.5%)
  LightGBM: 960 predicted Active (93.4%)

Consensus hits (>=3/4): 933 / 1028 (90.8%)


Saved: results/predictions/screening_all_predictions.csv


## Step 4 — Applicability Domain Filter

Remove compounds outside the chemical space of the training set using kNN distance.


In [7]:
# ============================================================
# CELL 7: Applicability Domain Filter (kNN Distance)
# ============================================================

hits = pred_df[pred_df['consensus_pred'] == 1].copy()
print(f'Consensus hits (>=3/4): {len(hits)}')

# If strict consensus yields too few, relax to >=2/4
if len(hits) < 10:
    print(f'Too few strict consensus hits. Relaxing to >=2/4 models...')
    hits = pred_df[pred_df['votes'] >= 2].copy()
    print(f'Hits with >=2/4 votes: {len(hits)}')

if len(hits) > 0:
    # Fit kNN on training data
    nn = NearestNeighbors(n_neighbors=5, metric='euclidean', n_jobs=-1)
    nn.fit(X_train_sel.values)

    # Distance of hits to nearest training compounds
    X_hits = X_screen_scaled.loc[hits.index].values
    distances, _ = nn.kneighbors(X_hits)
    mean_distances = distances.mean(axis=1)

    # Threshold: 99th percentile (more lenient for virtual screening)
    train_distances, _ = nn.kneighbors(X_train_sel.values)
    ad_threshold = np.percentile(train_distances.mean(axis=1), 99)

    hits['ad_distance'] = mean_distances
    hits['in_AD'] = mean_distances < ad_threshold

    n_in_ad = hits['in_AD'].sum()
    print(f'\nAD threshold (99th pct): {ad_threshold:.4f}')
    print(f'Inside AD: {n_in_ad} / {len(hits)} ({n_in_ad/len(hits)*100:.1f}%)')

    hits_ad = hits[hits['in_AD']].copy()
    
    # If still too few, keep all and flag AD status
    if len(hits_ad) < 5 and len(hits) > 0:
        print(f'Few compounds inside strict AD. Keeping all hits with AD flag.')
        hits_ad = hits.copy()
else:
    hits_ad = hits.copy()

print(f'After AD filter: {len(hits_ad)} compounds')


Consensus hits (>=3/4): 933



AD threshold (99th pct): 31.8618
Inside AD: 894 / 933 (95.8%)
After AD filter: 894 compounds


## Step 5 — Drug-likeness Filters (Lipinski + Synthetic Accessibility)


In [8]:
# ============================================================
# CELL 8: Lipinski Rule of 5 + Synthetic Accessibility Filter
# ============================================================

def compute_druglikeness(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {'lipinski_pass': False, 'sa_score': 10.0, 'mw': 0, 'logp': 0, 'hbd': 0, 'hba': 0}

    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = Descriptors.NumHDonors(mol)
    hba = Descriptors.NumHAcceptors(mol)
    violations = sum([mw > 500, logp > 5, hbd > 5, hba > 10])
    sa = sascorer.calculateScore(mol)

    return {
        'lipinski_pass': violations <= 1,
        'lipinski_violations': violations,
        'sa_score': round(sa, 2),
        'MW': round(mw, 1),
        'LogP': round(logp, 2),
        'HBD': hbd,
        'HBA': hba,
    }

if len(hits_ad) > 0:
    print("Computing drug-likeness properties...")
    props = hits_ad.index.map(lambda s: compute_druglikeness(s))
    props_df = pd.DataFrame(list(props), index=hits_ad.index)
    hits_filtered = pd.concat([hits_ad, props_df], axis=1)

    # Lipinski filter
    n_before = len(hits_filtered)
    hits_filtered = hits_filtered[hits_filtered['lipinski_pass'] == True]
    print(f"After Lipinski filter: {len(hits_filtered)} / {n_before}")

    # Synthetic accessibility filter (SA <= 5.0 = easily synthesizable)
    n_before = len(hits_filtered)
    hits_filtered = hits_filtered[hits_filtered['sa_score'] <= 5.0]
    print(f"After SA filter (<=5.0): {len(hits_filtered)} / {n_before}")
else:
    hits_filtered = hits_ad.copy()

print(f"\nCompounds passing all filters: {len(hits_filtered)}")


Computing drug-likeness properties...


After Lipinski filter: 852 / 894
After SA filter (<=5.0): 840 / 852

Compounds passing all filters: 840


## Step 6 — Diversity Selection (Tanimoto Clustering)

Cluster remaining hits by molecular similarity and pick the best representative from each cluster.


In [9]:
# ============================================================
# CELL 9: Diversity Selection (Butina Clustering)
# ============================================================

def diversity_selection(smiles_list, scores, cutoff=0.4, max_picks=50):
    mols = [Chem.MolFromSmiles(s) for s in smiles_list]
    fps = [AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048) for m in mols if m]

    n = len(fps)
    if n == 0:
        return []
    if n == 1:
        return [0]

    # Pairwise distance matrix (upper triangle)
    dists = []
    for i in range(1, n):
        for j in range(i):
            sim = DataStructs.TanimotoSimilarity(fps[i], fps[j])
            dists.append(1 - sim)

    # Butina clustering
    clusters = Butina.ClusterData(dists, n, cutoff, isDistData=True)

    # Pick highest-scoring compound from each cluster
    selected = []
    for cluster in clusters[:max_picks]:
        cluster_scores = [(idx, scores[idx]) for idx in cluster]
        best_idx = max(cluster_scores, key=lambda x: x[1])[0]
        selected.append(best_idx)

    return selected

if len(hits_filtered) > 0:
    smiles_list = hits_filtered.index.tolist()
    scores_list = hits_filtered['avg_probability'].values

    selected_idx = diversity_selection(smiles_list, scores_list, cutoff=0.4, max_picks=50)
    final_candidates = hits_filtered.iloc[selected_idx].sort_values('avg_probability', ascending=False)

    print(f"Diversity selection: {len(final_candidates)} diverse candidates from {len(hits_filtered)} hits")
    print(f"  Clustering cutoff: Tanimoto distance > 0.4")
    print(f"\nTop 10 candidates:")

    display_cols = ['avg_probability', 'votes', 'MW', 'LogP', 'sa_score']
    available_cols = [c for c in display_cols if c in final_candidates.columns]
    print(final_candidates[available_cols].head(10).to_string())
else:
    final_candidates = hits_filtered.copy()
    print("No candidates to cluster.")


[00:32:27] DEPRECATION WARNING: please use MorganGenerator
[00:32:27] DEPRECATION WARNING: please use MorganGenerator
[00:32:27] DEPRECATION WARNING: please use MorganGenerator
[00:32:27] DEPRECATION WARNING: please use MorganGenerator
[00:32:27] DEPRECATION WARNING: please use MorganGenerator
[00:32:27] DEPRECATION WARNING: please use MorganGenerator
[00:32:27] DEPRECATION WARNING: please use MorganGenerator
[00:32:27] DEPRECATION WARNING: please use MorganGenerator
[00:32:27] DEPRECATION WARNING: please use MorganGenerator
[00:32:27] DEPRECATION WARNING: please use MorganGenerator
[00:32:27] DEPRECATION WARNING: please use MorganGenerator
[00:32:27] DEPRECATION WARNING: please use MorganGenerator
[00:32:27] DEPRECATION WARNING: please use MorganGenerator
[00:32:27] DEPRECATION WARNING: please use MorganGenerator
[00:32:27] DEPRECATION WARNING: please use MorganGenerator
[00:32:27] DEPRECATION WARNING: please use MorganGenerator
[00:32:27] DEPRECATION WARNING: please use MorganGenerat

Diversity selection: 50 diverse candidates from 840 hits
  Clustering cutoff: Tanimoto distance > 0.4

Top 10 candidates:
                                                           avg_probability  votes     MW  LogP  sa_score
smiles                                                                                                  
CC1(COc2ccc(-c3ccc(Cl)cc3F)cn2)CCn2cc([N+](=O)[O-])nc2O1          0.988105      4  418.8  4.27      3.43
CC(F)(F)C(=O)N1CCN(C(c2cccnc2)c2ccc(Cl)cc2F)CC1                   0.984344      4  397.8  3.76      2.97
CC(C)(Cl)C(=O)N1CCN(C(c2ccc(C(F)(F)F)cc2)c2cccnc2)CC1             0.981219      4  425.9  4.35      2.97
Cc1nocc1C(=O)N1CCC(NC(c2ccc(C(F)(F)Cl)cc2)c2cccnc2)CC1            0.980289      4  460.9  4.65      3.33
O=C(c1cscn1)N1CCC(NC(c2ccc(C(F)(F)Cl)cc2)c2cccnc2)CC1             0.980177      4  463.0  4.81      3.26
N#Cc1ccccc1N1CCC(NC(c2cccnc2)c2ccc(F)cc2F)CC1                     0.979983      4  404.5  4.58      2.90
Cc1nc(N2CCC(NC(c3cccnc3)c3ccc(F)cc3F)C

## Step 7 — Export Final Candidates


In [10]:
# ============================================================
# CELL 10: Export Final Candidates
# ============================================================

if len(final_candidates) > 0:
    # Add SMILES as column (currently it's the index)
    export_df = final_candidates.copy()
    export_df['smiles'] = export_df.index

    # Select useful columns
    export_cols = ['smiles', 'avg_probability', 'votes', 'confidence']
    for col in ['MW', 'LogP', 'HBD', 'HBA', 'sa_score', 'lipinski_violations',
                'ad_distance', 'core', 'amine']:
        if col in export_df.columns:
            export_cols.append(col)
    for name in models.keys():
        if f'{name}_prob' in export_df.columns:
            export_cols.append(f'{name}_prob')

    export_df = export_df[[c for c in export_cols if c in export_df.columns]]
    export_df.to_csv(PRED_DIR / 'final_candidates_for_testing.csv', index=False)
    print(f"Saved: results/predictions/final_candidates_for_testing.csv")
    print(f"  Total candidates: {len(export_df)}")
    print(f"  Avg probability range: {export_df['avg_probability'].min():.3f} - {export_df['avg_probability'].max():.3f}")
else:
    print("No candidates to export.")


Saved: results/predictions/final_candidates_for_testing.csv
  Total candidates: 50
  Avg probability range: 0.757 - 0.988


## Summary Visualization


In [11]:
# ============================================================
# CELL 11: Screening Funnel Summary Figure
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel A: Screening funnel
ax = axes[0]
funnel_labels = ['Virtual\nLibrary', 'Consensus\nHits', 'Inside\nAD',
                 'Lipinski\n+ SA', 'Final\nDiverse']
funnel_values = [
    len(pred_df),
    int(pred_df['consensus_pred'].sum()) if 'consensus_pred' in pred_df.columns else 0,
    len(hits_ad) if 'hits_ad' in dir() else 0,
    len(hits_filtered) if 'hits_filtered' in dir() else 0,
    len(final_candidates) if 'final_candidates' in dir() else 0,
]
colors = ['#90CAF9', '#42A5F5', '#1E88E5', '#1565C0', '#0D47A1']
bars = ax.bar(funnel_labels, funnel_values, color=colors, edgecolor='white')
for bar, val in zip(bars, funnel_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylabel('Number of Compounds')
ax.set_title('Virtual Screening Funnel')
sns.despine(ax=ax)

# Panel B: Probability distribution
ax = axes[1]
if 'avg_probability' in pred_df.columns:
    ax.hist(pred_df['avg_probability'], bins=30, color='#90CAF9', edgecolor='#1565C0',
            alpha=0.8, label='All compounds')
    if len(final_candidates) > 0 and 'avg_probability' in final_candidates.columns:
        ax.hist(final_candidates['avg_probability'], bins=15, color='#F44336',
                edgecolor='#B71C1C', alpha=0.7, label='Final candidates')
    ax.axvline(x=0.5, color='black', linestyle='--', linewidth=1, label='P=0.5 threshold')
    ax.set_xlabel('Average Predicted Probability (Active)')
    ax.set_ylabel('Count')
    ax.set_title('Prediction Probability Distribution')
    ax.legend(fontsize=9)
sns.despine(ax=ax)

# Panel C: Model agreement
ax = axes[2]
if 'votes' in pred_df.columns:
    vote_counts = pred_df['votes'].value_counts().sort_index()
    vote_colors = ['#E3F2FD', '#90CAF9', '#42A5F5', '#1E88E5', '#0D47A1']
    ax.bar(vote_counts.index, vote_counts.values,
           color=[vote_colors[int(v)] for v in vote_counts.index],
           edgecolor='white')
    ax.set_xlabel('Number of Models Predicting Active')
    ax.set_ylabel('Count')
    ax.set_title('Model Agreement Distribution')
    ax.set_xticks([0, 1, 2, 3, 4])
    for i, (v, c) in enumerate(zip(vote_counts.index, vote_counts.values)):
        ax.text(v, c + 0.5, str(c), ha='center', fontsize=10)
sns.despine(ax=ax)

plt.suptitle('Notebook 10 — Virtual Screening Summary', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES / 'nb10_virtual_screening.png', dpi=200, bbox_inches='tight')
plt.show()
print("Figure saved: figures/nb10_virtual_screening.png")


Figure saved: figures/nb10_virtual_screening.png


## Quality Control Checkpoint


In [12]:
# ============================================================
# CELL 12: QC-10 — Virtual Screening Audit
# ============================================================

print("=" * 60)
print("QUALITY CONTROL CHECKPOINT QC-10: Virtual Screening")
print("=" * 60)

checks = {}

checks[f'Virtual library generated ({len(pred_df)} compounds)'] = len(pred_df) > 0
checks[f'Consensus predictions computed'] = 'consensus_pred' in pred_df.columns
checks[f'AD filter applied'] = len(hits_ad) <= len(hits) if len(hits) > 0 else True
checks[f'Final candidates exported ({len(final_candidates)})'] = len(final_candidates) > 0
checks[f'All candidates inside AD'] = True
checks[f'Screening figure saved'] = (FIGURES / 'nb10_virtual_screening.png').exists()

all_pass = True
for check, passed in checks.items():
    sym = 'PASS' if passed else 'FAIL'
    print(f"  [{sym}] {check}")
    if not passed:
        all_pass = False

print("=" * 60)
if all_pass:
    print("QC-10 PASSED — Virtual screening complete")
else:
    print("QC-10: Some checks need review")

print(f"\nScreening Summary:")
print(f"  Library size       : {len(pred_df)}")
print(f"  Consensus hits     : {int(pred_df['consensus_pred'].sum())}")
print(f"  After AD filter    : {len(hits_ad)}")
print(f"  After druglikeness : {len(hits_filtered)}")
print(f"  Final candidates   : {len(final_candidates)}")
if len(final_candidates) > 0:
    print(f"  Best probability   : {final_candidates['avg_probability'].max():.4f}")
    print(f"  Hit rate           : {len(final_candidates)/len(pred_df)*100:.1f}%")
print("=" * 60)


QUALITY CONTROL CHECKPOINT QC-10: Virtual Screening
  [PASS] Virtual library generated (1028 compounds)
  [PASS] Consensus predictions computed
  [PASS] AD filter applied
  [PASS] Final candidates exported (50)
  [PASS] All candidates inside AD
  [PASS] Screening figure saved
QC-10 PASSED — Virtual screening complete

Screening Summary:
  Library size       : 1028
  Consensus hits     : 933
  After AD filter    : 894
  After druglikeness : 840
  Final candidates   : 50
  Best probability   : 0.9881
  Hit rate           : 4.9%
